In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive



QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION


Purpose:
    Develop the final multi-label disease classification head using the
    locked Experiment E (Seed 42) bilateral fused representation.

Locked Upstream Configuration:
    • Experiment: E — Adaptive Bilateral Fusion
    • Seed: 42
    • Best Epoch: 5
    • Val Macro-F1: 0.561936
    • Val Micro-F1: 0.535966
    • Exact Match: 0.175799
    • Locked artifact: stage2_experiment_e_locked_seed42.pt

Strict Constraints:
    • Experiment E is FROZEN.
    • Cross-Eye representations are FROZEN.
    • Adaptive bilateral fusion is FROZEN.
    • No changes to the upstream backbone, disease attention,
      cross-eye attention, or fusion mechanism.
    • Only the downstream multi-label classification module will be optimized.

Classification Objective:
    Predict the 8 disease labels simultaneously from the locked fused
    bilateral representation.

Optimization Goal:
    Maximize multi-label diagnostic performance, with particular emphasis on:
    • Macro-F1
    • Micro-F1
    • Per-disease F1
    • Exact Match
    • Robust handling of class imbalance

Experimental Strategy:
    Start with strong, established multi-label classification approaches
    rather than weak baseline architectures or arbitrary architectural changes.

    Initial focus:
    • Strong classification head
    • Appropriate normalization and regularization
    • Class-imbalance-aware loss
    • Multi-label-specific loss functions
    • Validation-based threshold optimization
    • Per-disease performance analysis

Important:
    The classification module must consume the locked Experiment E
    representation directly. Upstream representations must not be retrained
    or modified during classifier development.



In [3]:
# =============================================================================
# QCDP-BiFormer — MODULE 9
# MULTI-LABEL CLASSIFICATION
# CELL 1 — ENVIRONMENT + LOCKED RESOURCES
# =============================================================================

import os
import random
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path

# -----------------------------------------------------------------------------
# 1. GOOGLE DRIVE
# -----------------------------------------------------------------------------

from google.colab import drive

drive.mount('/content/drive')

ROOT = Path("/content/drive/My Drive/Eye Disease/Dataset")

assert ROOT.exists(), f"Dataset root not found: {ROOT}"

print("=" * 80)
print("QCDP-BiFormer — MULTI-LABEL CLASSIFICATION")
print("=" * 80)
print(f"Dataset root: {ROOT}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# 2. REPRODUCIBILITY
# -----------------------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Deterministic behavior for evaluation/reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"\nGlobal seed: {SEED}")

# -----------------------------------------------------------------------------
# 3. LOCKED EXPERIMENT
# -----------------------------------------------------------------------------

LOCKED_EXPERIMENT = "experiment_e"
LOCKED_SEED = 42

LOCKED_ARTIFACT = ROOT / "stage2_experiment_e_locked_seed42.pt"

assert LOCKED_ARTIFACT.exists(), (
    f"Locked Experiment E artifact not found:\n{LOCKED_ARTIFACT}"
)

print("\nLocked configuration:")
print(f"  Experiment : {LOCKED_EXPERIMENT}")
print(f"  Seed       : {LOCKED_SEED}")
print(f"  Artifact   : {LOCKED_ARTIFACT}")

# -----------------------------------------------------------------------------
# 4. DATASET / LABEL DEFINITIONS
# -----------------------------------------------------------------------------

TRAIN_DF_PATH = ROOT / "stage2_train_df.csv"
VAL_DF_PATH   = ROOT / "val_patient_df.csv"

assert TRAIN_DF_PATH.exists(), f"Missing: {TRAIN_DF_PATH}"
assert VAL_DF_PATH.exists(), f"Missing: {VAL_DF_PATH}"

train_df = pd.read_csv(TRAIN_DF_PATH)
val_df   = pd.read_csv(VAL_DF_PATH)

# Fixed disease ordering used throughout QCDP-BiFormer
DISEASE_NAMES = [
    "N",  # Normal
    "D",  # Diabetic Retinopathy
    "G",  # Glaucoma
    "C",  # Cataract
    "A",  # AMD
    "H",  # Hypertension
    "M",  # Myopia
    "O",  # Other
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\nDataset:")
print(f"  Train samples : {len(train_df)}")
print(f"  Val samples   : {len(val_df)}")
print(f"  Classes       : {NUM_CLASSES}")
print(f"  Label order   : {DISEASE_NAMES}")

# -----------------------------------------------------------------------------
# 5. EXISTING LOCKED / UPSTREAM RESOURCES
# -----------------------------------------------------------------------------

RESOURCE_FILES = {
    "stage1_train_df": ROOT / "stage1_train_df.csv",
    "stage2_train_df": ROOT / "stage2_train_df.csv",
    "stage1_quality_scores": ROOT / "stage1_quality_scores.csv",

    "disease_aware_features": ROOT / "disease_aware_features.pt",
    "disease_attention_weights": ROOT / "disease_attention_weights.pt",
    "disease_prototypes": ROOT / "disease_prototypes.pt",

    "cross_eye_train": ROOT / "stage2_train_cross_eye_features_final.pt",
    "cross_eye_val": ROOT / "stage2_val_cross_eye_features_final.pt",

    "locked_experiment_e": LOCKED_ARTIFACT,
}

print("\nUpstream resources:")
for name, path in RESOURCE_FILES.items():
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"  [{status:>9}] {name}: {path.name}")

# -----------------------------------------------------------------------------
# 6. LOAD LOCKED EXPERIMENT E ARTIFACT
# -----------------------------------------------------------------------------

print("\n" + "-" * 80)
print("Loading locked Experiment E artifact...")
print("-" * 80)

locked_artifact = torch.load(
    LOCKED_ARTIFACT,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(locked_artifact)}")

# Inspect structure without modifying anything
if isinstance(locked_artifact, dict):

    print("\nArtifact contents:")
    for key, value in locked_artifact.items():

        if torch.is_tensor(value):
            print(
                f"  {key}: Tensor "
                f"shape={tuple(value.shape)}, "
                f"dtype={value.dtype}"
            )

        elif isinstance(value, dict):
            print(
                f"  {key}: dict "
                f"({len(value)} entries)"
            )

        elif isinstance(value, (list, tuple)):
            print(
                f"  {key}: {type(value).__name__} "
                f"({len(value)} entries)"
            )

        else:
            print(
                f"  {key}: "
                f"{type(value).__name__} = {value}"
            )

# -----------------------------------------------------------------------------
# 7. LOCKED REFERENCE METRICS
# -----------------------------------------------------------------------------

LOCKED_REFERENCE = {
    "best_epoch": 5,
    "val_loss": 0.791240,
    "micro_f1": 0.535966,
    "macro_f1": 0.561936,
    "exact_match": 0.175799,
}

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E / SEED 42 REFERENCE")
print("-" * 80)

for metric, value in LOCKED_REFERENCE.items():
    print(f"{metric:>15}: {value}")

# -----------------------------------------------------------------------------
# 8. GLOBAL CLASSIFICATION CONFIGURATION
# -----------------------------------------------------------------------------

FEATURE_DIM = 768
NUM_CLASSES = 8

# We will NOT modify these upstream representations.
UPSTREAM_FROZEN = True

print("\n" + "=" * 80)
print("MODULE 9 INITIALIZATION COMPLETE")
print("=" * 80)

print(f"""
Locked Experiment       : E
Locked Seed             : 42
Feature dimension       : {FEATURE_DIM}
Number of diseases      : {NUM_CLASSES}
Disease ordering        : {DISEASE_NAMES}

Upstream fusion         : FROZEN
Cross-Eye representations: FROZEN
Adaptive fusion         : FROZEN
Mean-fusion replacement : NOT ALLOWED

Next step:
    Inspect the artifact structure and determine exactly which
    fused representation should enter the classification head.
""")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
QCDP-BiFormer — MULTI-LABEL CLASSIFICATION
Dataset root: /content/drive/My Drive/Eye Disease/Dataset
PyTorch version: 2.11.0+cpu
CUDA available: False

Global seed: 42

Locked configuration:
  Experiment : experiment_e
  Seed       : 42
  Artifact   : /content/drive/My Drive/Eye Disease/Dataset/stage2_experiment_e_locked_seed42.pt

Dataset:
  Train samples : 2141
  Val samples   : 504
  Classes       : 8
  Label order   : ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Upstream resources:
  [    FOUND] stage1_train_df: stage1_train_df.csv
  [    FOUND] stage2_train_df: stage2_train_df.csv
  [    FOUND] stage1_quality_scores: stage1_quality_scores.csv
  [    FOUND] disease_aware_features: disease_aware_features.pt
  [    FOUND] disease_attention_weights: disease_attention_weights.pt
  [    FOUND] disease_prototypes: disease_prototypes.pt
  [    FOUND] cross_eye_trai

CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS


Objective:
    Load the officially exported Experiment E / Seed 42 fused representations
    and verify their integrity before designing and training the multi-label
    classification head.

OFFICIALLY LOCKED UPSTREAM:
    Experiment E — Strict Class-wise Adaptive Fusion
    Seed: 42

Representation:
    Train: [2141, 8, 768]
    Val  : [438, 8, 768]

Interpretation:
    Each patient has 8 disease-specific fused representations, with each
    disease receiving its own 768-dimensional representation.

Disease order:
    ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']

Disease mapping:
    N = Normal
    D = Diabetic Retinopathy
    G = Glaucoma
    C = Cataract
    A = Age-related Macular Degeneration
    H = Hypertension
    M = Myopia
    O = Other

Validation cohort:
    The 438-patient bilateral cohort is used because it is the exact cohort
    on which the locked Experiment E representation was generated.

STRICT UPSTREAM LOCK:
    • No fusion retraining
    • No feature modification
    • No new patient filtering
    • No new train/validation split
    • No mean-fusion replacement
    • No changes to Experiment E

This cell performs verification only.

Next:
    Determine the most appropriate classification-head input strategy for
    the disease-specific [8 × 768] representation before beginning
    optimization experiments.


In [8]:
# =============================================================================
# QCDP-BiFormer — MODULE 9: MULTI-LABEL CLASSIFICATION
# CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS
# =============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print("=" * 80)
print("CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS")
print("=" * 80)


# =============================================================================
# 1. PATHS
# =============================================================================

ROOT = Path(
    "/content/drive/My Drive/Eye Disease/Dataset"
)

FUSED_ARTIFACT_PATH = (
    ROOT / "stage2_experiment_e_fused_representations_seed42.pt"
)

TRAIN_LABEL_PATH = (
    ROOT / "stage2_train_bilateral_metadata.csv"
)

VAL_LABEL_PATH = (
    ROOT / "stage2_val_bilateral_metadata.csv"
)

assert FUSED_ARTIFACT_PATH.exists(), (
    f"Missing fused representation artifact:\n{FUSED_ARTIFACT_PATH}"
)

assert TRAIN_LABEL_PATH.exists(), (
    f"Missing train bilateral metadata:\n{TRAIN_LABEL_PATH}"
)

assert VAL_LABEL_PATH.exists(), (
    f"Missing validation bilateral metadata:\n{VAL_LABEL_PATH}"
)

print(f"\nDataset root: {ROOT}")
print(f"Fused artifact: {FUSED_ARTIFACT_PATH.name}")


# =============================================================================
# 2. LOAD ARTIFACT
# =============================================================================

print("\n" + "-" * 80)
print("LOADING LOCKED FUSED REPRESENTATIONS")
print("-" * 80)

fused_artifact = torch.load(
    FUSED_ARTIFACT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Artifact loaded successfully.")
print(f"Artifact type: {type(fused_artifact)}")


# =============================================================================
# 3. VERIFY LOCKED PROVENANCE
# =============================================================================

print("\n" + "-" * 80)
print("VERIFYING LOCKED PROVENANCE")
print("-" * 80)

print(
    f"Experiment     : "
    f"{fused_artifact.get('experiment')}"
)

print(
    f"Experiment key : "
    f"{fused_artifact.get('experiment_key')}"
)

print(
    f"Seed           : "
    f"{fused_artifact.get('seed')}"
)

print(
    f"Status         : "
    f"{fused_artifact.get('status')}"
)

assert fused_artifact["experiment_key"] == "experiment_e"
assert fused_artifact["seed"] == 42
assert fused_artifact["status"] == "LOCKED"

print("\n✓ Experiment E verified.")
print("✓ Seed 42 verified.")
print("✓ Artifact status = LOCKED.")


# =============================================================================
# 4. DISEASE ORDER
# =============================================================================

DISEASE_NAMES = [
    "N",
    "D",
    "G",
    "C",
    "A",
    "H",
    "M",
    "O",
]

NUM_CLASSES = len(DISEASE_NAMES)

print("\n" + "-" * 80)
print("DISEASE ORDER")
print("-" * 80)

print(DISEASE_NAMES)

artifact_labels = list(
    fused_artifact["label_columns"]
)

print(
    f"\nArtifact label order: {artifact_labels}"
)

assert artifact_labels == DISEASE_NAMES, (
    "Artifact disease ordering does not match Module 9 ordering."
)

print("✓ Disease ordering verified.")


# =============================================================================
# 5. EXTRACT FUSED FEATURES
# =============================================================================

print("\n" + "-" * 80)
print("EXTRACTING FUSED FEATURES")
print("-" * 80)

X_train = fused_artifact[
    "train_classwise_features"
].float()

X_val = fused_artifact[
    "val_classwise_features"
].float()


# -----------------------------------------------------------------------------
# IMPORTANT:
# The fused artifact stores patient IDs as 0-D PyTorch tensors:
#     tensor(1), tensor(4), ...
#
# The CSV stores them as integers:
#     1, 4, ...
#
# Normalize them here so the remainder of the notebook works with ordinary
# Python integer patient IDs.
# -----------------------------------------------------------------------------

def normalize_patient_ids(ids):

    normalized = []

    for x in ids:

        if torch.is_tensor(x):
            normalized.append(int(x.item()))

        else:
            normalized.append(int(x))

    return normalized


train_patient_ids = normalize_patient_ids(
    fused_artifact["train_patient_ids"]
)

val_patient_ids = normalize_patient_ids(
    fused_artifact["val_patient_ids"]
)

print(
    f"Train fused representation : "
    f"{tuple(X_train.shape)}"
)

print(
    f"Val fused representation   : "
    f"{tuple(X_val.shape)}"
)

print(
    f"Train patient IDs          : "
    f"{len(train_patient_ids)}"
)

print(
    f"Val patient IDs            : "
    f"{len(val_patient_ids)}"
)

print(
    f"\nFirst 10 normalized train IDs:"
)

print(train_patient_ids[:10])


# =============================================================================
# 6. STRICT SHAPE VERIFICATION
# =============================================================================

assert X_train.ndim == 3
assert X_val.ndim == 3

assert X_train.shape[1] == NUM_CLASSES
assert X_val.shape[1] == NUM_CLASSES

assert X_train.shape[2] == 768
assert X_val.shape[2] == 768

assert X_train.shape[0] == len(train_patient_ids)
assert X_val.shape[0] == len(val_patient_ids)

print("\n✓ Train shape = [2141, 8, 768]")
print("✓ Validation shape = [438, 8, 768]")
print("✓ Eight disease-specific representations confirmed.")
print("✓ Representation dimension = 768.")


# =============================================================================
# 7. CHECK FINITE VALUES
# =============================================================================

print("\n" + "-" * 80)
print("NUMERICAL INTEGRITY")
print("-" * 80)

train_nan = torch.isnan(X_train).sum().item()
train_inf = torch.isinf(X_train).sum().item()

val_nan = torch.isnan(X_val).sum().item()
val_inf = torch.isinf(X_val).sum().item()

print(f"Train NaN values : {train_nan}")
print(f"Train Inf values : {train_inf}")
print(f"Val NaN values   : {val_nan}")
print(f"Val Inf values   : {val_inf}")

assert train_nan == 0
assert train_inf == 0
assert val_nan == 0
assert val_inf == 0

print("\n✓ No NaN values.")
print("✓ No infinite values.")


# =============================================================================
# 8. REPRESENTATION STATISTICS
# =============================================================================

print("\n" + "-" * 80)
print("REPRESENTATION STATISTICS")
print("-" * 80)

print(
    f"Train mean : {X_train.mean().item():.6f}"
)

print(
    f"Train std  : {X_train.std().item():.6f}"
)

print(
    f"Train min  : {X_train.min().item():.6f}"
)

print(
    f"Train max  : {X_train.max().item():.6f}"
)

print()

print(
    f"Val mean   : {X_val.mean().item():.6f}"
)

print(
    f"Val std    : {X_val.std().item():.6f}"
)

print(
    f"Val min    : {X_val.min().item():.6f}"
)

print(
    f"Val max    : {X_val.max().item():.6f}"
)


# =============================================================================
# 9. LOAD BILATERAL LABEL DATA
# =============================================================================

print("\n" + "-" * 80)
print("LOADING BILATERAL LABEL DATA")
print("-" * 80)

train_df = pd.read_csv(
    TRAIN_LABEL_PATH
)

val_df = pd.read_csv(
    VAL_LABEL_PATH
)

print(
    f"Train dataframe: {train_df.shape}"
)

print(
    f"Validation dataframe: {val_df.shape}"
)


# =============================================================================
# 10. VERIFY LABEL COLUMNS
# =============================================================================

missing_train = [
    disease for disease in DISEASE_NAMES
    if disease not in train_df.columns
]

missing_val = [
    disease for disease in DISEASE_NAMES
    if disease not in val_df.columns
]

assert not missing_train, (
    f"Missing train labels: {missing_train}"
)

assert not missing_val, (
    f"Missing validation labels: {missing_val}"
)

print("\n✓ All eight disease labels present.")


# =============================================================================
# 11. VERIFY PATIENT COUNTS
# =============================================================================

assert len(train_df) == X_train.shape[0]
assert len(val_df) == X_val.shape[0]

print("\n" + "-" * 80)
print("PATIENT COUNT ALIGNMENT")
print("-" * 80)

print(
    f"Train features : {len(X_train)}"
)

print(
    f"Train labels   : {len(train_df)}"
)

print(
    f"Val features   : {len(X_val)}"
)

print(
    f"Val labels     : {len(val_df)}"
)

print("\n✓ Train counts aligned.")
print("✓ Validation counts aligned.")


# =============================================================================
# 12. VERIFY PATIENT-ID ALIGNMENT
# =============================================================================

print("\n" + "-" * 80)
print("PATIENT-ID ALIGNMENT")
print("-" * 80)

possible_id_columns = [
    "patient_id",
    "Patient_ID",
    "patientID",
    "PatientID",
    "patient",
    "ID",
    "id",
]

train_id_candidates = [
    c for c in possible_id_columns
    if c in train_df.columns
]

val_id_candidates = [
    c for c in possible_id_columns
    if c in val_df.columns
]

print(
    f"Train ID candidates: {train_id_candidates}"
)

print(
    f"Val ID candidates  : {val_id_candidates}"
)

assert len(train_id_candidates) == 1, (
    "Could not uniquely identify train patient-ID column."
)

assert len(val_id_candidates) == 1, (
    "Could not uniquely identify validation patient-ID column."
)

train_id_col = train_id_candidates[0]
val_id_col = val_id_candidates[0]

train_df_ids = [
    int(x)
    for x in train_df[train_id_col].tolist()
]

val_df_ids = [
    int(x)
    for x in val_df[val_id_col].tolist()
]

assert len(train_df_ids) == len(set(train_df_ids))
assert len(val_df_ids) == len(set(val_df_ids))

train_feature_set = set(train_patient_ids)
train_label_set = set(train_df_ids)

val_feature_set = set(val_patient_ids)
val_label_set = set(val_df_ids)

assert train_feature_set == train_label_set, (
    "Train patient IDs do not match exactly."
)

assert val_feature_set == val_label_set, (
    "Validation patient IDs do not match exactly."
)

print("\n✓ Train patient IDs match exactly.")
print("✓ Validation patient IDs match exactly.")


# =============================================================================
# 13. BUILD LABEL MATRICES
# =============================================================================

print("\n" + "-" * 80)
print("BUILDING MULTI-LABEL TARGET MATRICES")
print("-" * 80)

# Reorder dataframes explicitly according to the fused-feature patient order.

train_lookup = train_df.set_index(train_id_col)

val_lookup = val_df.set_index(val_id_col)

train_ordered_df = train_lookup.loc[
    train_patient_ids
].reset_index()

val_ordered_df = val_lookup.loc[
    val_patient_ids
].reset_index()

Y_train = torch.tensor(
    train_ordered_df[DISEASE_NAMES].values,
    dtype=torch.float32
)

Y_val = torch.tensor(
    val_ordered_df[DISEASE_NAMES].values,
    dtype=torch.float32
)

print(
    f"Y_train shape: {tuple(Y_train.shape)}"
)

print(
    f"Y_val shape  : {tuple(Y_val.shape)}"
)

assert Y_train.shape == (
    X_train.shape[0],
    NUM_CLASSES
)

assert Y_val.shape == (
    X_val.shape[0],
    NUM_CLASSES
)

print("\n✓ Feature → patient → label ordering verified.")


# =============================================================================
# 14. VERIFY BINARY LABELS
# =============================================================================

unique_train_labels = torch.unique(Y_train).tolist()
unique_val_labels = torch.unique(Y_val).tolist()

print("\nTrain unique label values:", unique_train_labels)
print("Val unique label values  :", unique_val_labels)

assert set(unique_train_labels).issubset({0.0, 1.0})
assert set(unique_val_labels).issubset({0.0, 1.0})

print("\n✓ Targets are binary multi-label indicators.")


# =============================================================================
# 15. CLASS DISTRIBUTION
# =============================================================================

print("\n" + "-" * 80)
print("CLASS DISTRIBUTION")
print("-" * 80)

print(
    f"{'Disease':>8} | "
    f"{'Train +':>8} | "
    f"{'Train %':>8} | "
    f"{'Val +':>8} | "
    f"{'Val %':>8}"
)

print("-" * 55)

for i, disease in enumerate(DISEASE_NAMES):

    train_pos = int(Y_train[:, i].sum().item())
    val_pos = int(Y_val[:, i].sum().item())

    train_prev = train_pos / len(Y_train)
    val_prev = val_pos / len(Y_val)

    print(
        f"{disease:>8} | "
        f"{train_pos:8d} | "
        f"{train_prev:8.4f} | "
        f"{val_pos:8d} | "
        f"{val_prev:8.4f}"
    )


# =============================================================================
# 16. MULTI-LABEL CARDINALITY
# =============================================================================

train_cardinality = Y_train.sum(dim=1)
val_cardinality = Y_val.sum(dim=1)

print("\n" + "-" * 80)
print("MULTI-LABEL CARDINALITY")
print("-" * 80)

print(
    f"Train mean labels/patient : "
    f"{train_cardinality.mean().item():.4f}"
)

print(
    f"Val mean labels/patient   : "
    f"{val_cardinality.mean().item():.4f}"
)

print(
    f"Train max labels/patient  : "
    f"{int(train_cardinality.max().item())}"
)

print(
    f"Val max labels/patient    : "
    f"{int(val_cardinality.max().item())}"
)


# =============================================================================
# 17. VERIFY LOCKED REFERENCE
# =============================================================================

print("\n" + "-" * 80)
print("LOCKED EXPERIMENT E REFERENCE")
print("-" * 80)

locked_reference = fused_artifact[
    "locked_reference"
]

for key, value in locked_reference.items():
    print(
        f"{key:20s}: {value}"
    )

assert locked_reference["best_epoch"] == 5

assert abs(
    locked_reference["val_micro_f1"] - 0.535966
) < 1e-6

assert abs(
    locked_reference["val_macro_f1"] - 0.561936
) < 1e-6

assert abs(
    locked_reference["exact_match"] - 0.175799
) < 1e-6

print("\n✓ Locked Experiment E reference verified.")


# =============================================================================
# 18. FINAL MODULE 9 DATA OBJECTS
# =============================================================================

print("\n" + "=" * 80)
print("CELL 2 COMPLETE")
print("=" * 80)

print("""
UPSTREAM HANDOFF VERIFIED

Train:
    X_train : [2141, 8, 768]
    Y_train : [2141, 8]

Validation:
    X_val   : [438, 8, 768]
    Y_val   : [438, 8]

Representation:
    Eight disease-specific 768-D fused representations per patient.

Patient alignment:
    ✓ Verified
    ✓ Tensor IDs normalized to integer IDs

Labels:
    ✓ Binary multi-label
    ✓ Correct disease ordering

Experiment E:
    ✓ Locked
    ✓ Seed 42
    ✓ No upstream modification

IMPORTANT:
    No classifier has been trained yet.

The next step is architectural design of the multi-label classification
head using the locked disease-specific fused representations.
""")

CELL 2 — LOAD & VERIFY LOCKED EXPERIMENT E REPRESENTATIONS

Dataset root: /content/drive/My Drive/Eye Disease/Dataset
Fused artifact: stage2_experiment_e_fused_representations_seed42.pt

--------------------------------------------------------------------------------
LOADING LOCKED FUSED REPRESENTATIONS
--------------------------------------------------------------------------------
Artifact loaded successfully.
Artifact type: <class 'dict'>

--------------------------------------------------------------------------------
VERIFYING LOCKED PROVENANCE
--------------------------------------------------------------------------------
Experiment     : Experiment E — Strict Class-wise Adaptive Fusion
Experiment key : experiment_e
Seed           : 42
Status         : LOCKED

✓ Experiment E verified.
✓ Seed 42 verified.
✓ Artifact status = LOCKED.

--------------------------------------------------------------------------------
DISEASE ORDER
-----------------------------------------------------